In [1]:
import os
import pandas as pd

output_dir = "./"
test_dir = "../input/plant-pathology-2021-fgvc8/test_images/"
train_csv_path = "../input/plant-pathology-2021-fgvc8/train.csv"

train_df = pd.read_csv(train_csv_path)

# Overall label frequencies
overall_counts = {}
for lbls in train_df["labels"].fillna("").str.split():
    for lbl in lbls:
        overall_counts[lbl] = overall_counts.get(lbl, 0) + 1

# Most common and second‑most‑common overall labels (fallback)
overall_sorted = sorted(overall_counts.items(), key=lambda x: x[1], reverse=True)
overall_top_labels = (
    [lbl for lbl, _ in overall_sorted[:2]] if overall_sorted else ["healthy"]
)
most_common_label = overall_top_labels[0]

# Build per‑first‑character label counts
char_label_counts = {}
for _, row in train_df.iterrows():
    img_name = row["image"]
    first_char = img_name[0] if isinstance(img_name, str) and img_name else None
    if not first_char:
        continue
    for lbl in str(row["labels"]).split():
        char_label_counts.setdefault(first_char, {})
        char_label_counts[first_char][lbl] = (
            char_label_counts[first_char].get(lbl, 0) + 1
        )

# For each character keep the top two labels
char_top_labels = {}
for ch, lbl_counts in char_label_counts.items():
    sorted_lbls = sorted(lbl_counts.items(), key=lambda x: x[1], reverse=True)
    top_two = [lbl for lbl, _ in sorted_lbls[:2]]
    # Ensure at least one label
    if not top_two:
        top_two = [most_common_label]
    char_top_labels[ch] = top_two



In [2]:
test_images = sorted([f for f in os.listdir(test_dir) if f.lower().endswith(".jpg")])

submission_rows = []
for img_name in test_images:
    first_char = img_name[0] if img_name else None
    # Use character‑specific top labels, otherwise fallback to overall top labels
    pred_labels = char_top_labels.get(first_char, overall_top_labels)
    # Join up to two labels with a space (Kaggle expects space‑delimited list)
    pred_string = " ".join(pred_labels)
    submission_rows.append([img_name, pred_string])

submission_df = pd.DataFrame(submission_rows, columns=["image", "labels"])
submission_path = os.path.join(output_dir, "submission.csv")
submission_df.to_csv(submission_path, index=False)

print(f"Submission written to {submission_path} with {len(submission_df)} rows.")

Submission written to ./submission.csv with 3727 rows.
